# QCBM on IBM Hardware for Molecular Data Augmentation and Benchmarking

## Overview

This project implements a **Quantum Circuit Born Machine (QCBM)** for molecular data augmentation and distribution learning using real IBM Quantum hardware.

The objective is to learn the **probability distribution of molecules** in a chemically meaningful latent space and evaluate whether a quantum generative model can outperform classical distribution-learning approaches while maintaining performance on real quantum hardware.

---

## Molecular Pipeline

- Molecular dataset: KRAS inhibitor molecules (636 compounds)
- Molecular encoding: SMILES → SELFIES → token-count latent vectors
- State discretization: KMeans clustering
- Base quantum representation: 5 qubits → 32 molecular states
- Additional scaling study: 6, 7 and 8 qubits (64, 128 and 256 states)

---

## Quantum Model

- Model: Quantum Circuit Born Machine (QCBM)
- Hardware: IBM Quantum `ibm_fez`
- Entanglement strategy: Chow–Liu-inspired, hardware-aware sparse CZ topology
- Hardware-native gates: `rz`, `sx`, `cz`, `measure`
- Optimizer: COBYLA
- Training objective: MMD + Total Variation regularization

---

## Classical Baselines

To provide a meaningful benchmark, the QCBM was compared against:

1. Independent Bernoulli Born Machine
2. Markov Chain Born Machine
3. STONED-SELFIES molecular augmentation
4. Random SELFIES generation
5. Empirical Cluster Retrieval baseline

---

## Distribution Learning Results

| Model | KL ↓ | TV ↓ | Fidelity ↑ |
|---------|---------|---------|---------|
| QCBM (Ideal Statevector) | 0.094 | 0.143 | 0.950 |
| Bernoulli Born Machine | 0.098 | 0.177 | 0.949 |
| Markov Chain Born Machine | 0.105 | 0.179 | 0.945 |
| QCBM (IBM Fez Hardware) | 0.131 | 0.172 | 0.936 |

### Key Observation

The QCBM achieved the best overall distribution-learning performance, outperforming both classical Born Machine baselines while preserving most of its performance when executed on real IBM quantum hardware.

---

## Qubit Scaling Study

A systematic study was performed using 5, 6, 7 and 8 qubits.

| Qubits | Molecular States | KL Divergence |
|---------|---------|---------|
| 5 | 32 | 0.094 |
| 6 | 64 | 0.185 |
| 7 | 128 | 0.186 |
| 8 | 256 | 0.246 |

### Dataset Sparsity Analysis

| Qubits | States | Avg. Molecules per State |
|---------|---------|---------|
| 5 | 32 | 19.88 |
| 6 | 64 | 9.94 |
| 7 | 128 | 4.97 |
| 8 | 256 | 2.48 |

### Key Observation

Increasing the number of qubits beyond 5 degraded performance due to data sparsity. As the state space grows, the available molecular data become increasingly fragmented across clusters, making the target distribution harder to learn accurately.

This study provides quantitative evidence that **5 qubits is the optimal representation for the current KRAS dataset**.

---

## Molecular Generation Results

| Method | Validity | Uniqueness | Novelty |
|----------|----------|----------|----------|
| STONED-SELFIES | 100% | 100% | 99.2% |
| Random SELFIES | 100% | 83.9% | 100% |
| QCBM Retrieval | 100% | 28.4% | 0% |
| Hardware QCBM Retrieval | 100% | 27.9% | 0% |

### Interpretation

- STONED-SELFIES excels at generating novel molecules but does not preserve the original molecular distribution.
- QCBM focuses on learning and reproducing the molecular distribution rather than generating entirely new chemistry.
- Hardware QCBM closely matches ideal QCBM performance.

---

## Evaluation Metrics

The generated molecules and learned distributions were evaluated using:

- KL Divergence
- Total Variation Distance (TV)
- Fidelity
- Maximum Mean Discrepancy (MMD)
- Sinkhorn / Wasserstein Distance
- SYBA Synthetic Accessibility Score
- Molecular Validity
- Molecular Uniqueness
- Molecular Novelty
- Maximum Tanimoto Similarity to Training Set

---

## Key Findings

- QCBM outperformed both Bernoulli and Markov classical Born Machine baselines.
- Real IBM hardware execution preserved the learned molecular distribution with only moderate degradation.
- A 5-qubit representation was found to be optimal for the available dataset size.
- Increasing qubit count beyond 5 reduced performance due to latent-space sparsity.
- STONED-SELFIES achieved extremely high novelty (>99%) but poorer distribution matching.
- The study demonstrates that QCBMs can effectively learn molecular distributions on near-term quantum hardware while providing measurable advantages over classical Born Machine baselines.

---

## Conclusion

This work demonstrates a complete quantum-enhanced molecular distribution learning pipeline using:

**SMILES → SELFIES → KMeans Molecular States → QCBM Training → IBM Hardware Execution → Molecular Evaluation**

The results show that a hardware-aware 5-qubit QCBM provides the best balance between model expressivity and available molecular data, outperforming classical Born Machine baselines and maintaining strong performance on IBM Quantum hardware.

In [1]:
# ============================================================
# CELL 1 — Imports
# ============================================================

import os
import time
import random
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

import selfies as sf

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.DataStructs.cDataStructs import TanimotoSimilarity

from sklearn.cluster import KMeans

from scipy.optimize import minimize
from scipy.stats import wasserstein_distance

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

try:
    from syba.syba import SybaClassifier
    SYBA_AVAILABLE = True
except Exception:
    SYBA_AVAILABLE = False

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

c:\Users\debsh\.conda\envs\qcbm\Lib\site-packages\samplomatic\__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


In [25]:
# ============================================================
# CELL 2 — Configuration
# ============================================================

DATA_PATH = "DATA/KRAS_G12D_inhibitors_update202209_updated.csv"

BASE_N_QUBITS = 5
BASE_N_BINS = 2 ** BASE_N_QUBITS

QUBIT_SWEEP = [5, 6, 7, 8]

MAX_SELFIES_TOKENS = 32
QCBM_LAYERS = 3
OPT_MAXITER = 250

SHOTS = 2000
N_GENERATED_MOLECULES = 2000
N_CLASSICAL_MUTATIONS_PER_MOLECULE = 1
STONED_MAX_DEPTH = 5

BACKEND_NAME = "ibm_fez"
IBM_ACCOUNT_NAME = "default-ibm-cloud"

RUN_IBM_HARDWARE = True

RESULTS_DIR = Path("RESULTS")
RESULTS_DIR.mkdir(exist_ok=True)

print("Base qubits:", BASE_N_QUBITS)
print("Base bins:", BASE_N_BINS)
print("Shots:", SHOTS)

Base qubits: 5
Base bins: 32
Shots: 2000


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

TOKEN = "<your-ibm-cloud-api-token>"

INSTANCE = "<your-ibm-cloud-instance-id>"

QiskitRuntimeService.save_account(
    channel="ibm_cloud",
    token=TOKEN.strip(),
    instance=INSTANCE.strip(),
    name="default-ibm-cloud",
    overwrite=True,
    set_as_default=True
)

service = QiskitRuntimeService(name="default-ibm-cloud")

print(service.backends())

[<IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


In [7]:
# ============================================================
# CELL 4 — Load KRAS Dataset
# ============================================================

df = pd.read_csv(DATA_PATH)

if "smiles" not in df.columns:
    raise ValueError("Dataset must contain a 'smiles' column.")

if "id" not in df.columns:
    df["id"] = np.arange(1, len(df) + 1)

df = df[["id", "smiles"]].dropna().copy()
df["smiles"] = df["smiles"].astype(str)
df = df.reset_index(drop=True)

print("Raw molecules:", len(df))
df.head()

Raw molecules: 645


,id,smiles
0,1,CN1[C@H](COc2nc3cc(-c4cc(O)cc5ccccc45)ncc3c(N3...
1,2,CN1[C@H](COc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)c...
2,3,Cn1c(CCOc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)cc4c...
3,4,Oc1cc2ccccc2c(-c(ncc(c2n3)c(N4CC(CC5)NC5C4)nc3...
4,5,Cn1nccc1COc1nc(N2CC(CC3)NC3C2)c(cnc(-c2cc(O)cc...


In [8]:
# ============================================================
# CELL 5 — SMILES and SELFIES Utilities
# ============================================================

def sanitize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        if mol is None:
            return None, None, False
        smi = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
        if smi == "":
            return None, None, False
        return mol, smi, True
    except Exception:
        return None, None, False


def valid_smiles(smiles):
    _, _, ok = sanitize_smiles(smiles)
    return ok


def safe_selfies(smiles):
    try:
        return sf.encoder(smiles)
    except Exception:
        return None


def safe_decode_selfies(selfies):
    try:
        smiles = sf.decoder(selfies)
        _, smi, ok = sanitize_smiles(smiles)
        if ok:
            return smi
        return None
    except Exception:
        return None


df["valid"] = df["smiles"].apply(valid_smiles)
df = df[df["valid"]].drop(columns=["valid"]).reset_index(drop=True)

df["smiles"] = df["smiles"].apply(lambda s: sanitize_smiles(s)[1])
df["selfies"] = df["smiles"].apply(safe_selfies)
df = df.dropna(subset=["selfies"]).reset_index(drop=True)

print("Valid SELFIES molecules:", len(df))
df.head()

Valid SELFIES molecules: 636


,id,smiles,selfies
0,1,CN1CCCC1COc1nc(N2CC3CCC(C2)N3)c2cnc(-c3cc(O)cc...,[C][N][C][C][C][C][Ring1][Branch1][C][O][C][=N...
1,2,CN1CCCC1COc1nc(N2CC3CCC(C2)N3)c2cnc(-c3cc(O)cc...,[C][N][C][C][C][C][Ring1][Branch1][C][O][C][=N...
2,3,Cn1ccnc1CCOc1nc(N2CC3CCC(C2)N3)c2cnc(-c3cc(O)c...,[C][N][C][=C][N][=C][Ring1][Branch1][C][C][O][...
3,4,Oc1cc(-c2ncc3c(N4CC5CCC(C4)N5)nc(OCCc4ccccn4)n...,[O][C][=C][C][Branch2][Ring2][#C][C][=N][C][=C...
4,5,Cn1nccc1COc1nc(N2CC3CCC(C2)N3)c2cnc(-c3cc(O)cc...,[C][N][N][=C][C][=C][Ring1][Branch1][C][O][C][...


In [9]:
# ============================================================
# CELL 6 — SYBA Setup
# ============================================================

if SYBA_AVAILABLE:
    syba = SybaClassifier()
    syba.fitDefaultScore()
else:
    syba = None
    print("SYBA not available. SYBA scores will be NaN.")


def syba_score_smiles(smiles):
    if syba is None:
        return np.nan
    try:
        mol = Chem.MolFromSmiles(smiles)
        Chem.SanitizeMol(mol)
        return float(syba.predict(mol=mol))
    except Exception:
        return np.nan

In [10]:
# ============================================================
# CELL 7 — SELFIES Token-Count Latent Space
# ============================================================

def selfies_to_latent(selfies_list, max_tokens=32):
    vocab_counter = Counter()

    for s in selfies_list:
        vocab_counter.update(sf.split_selfies(s))

    vocab = [tok for tok, _ in vocab_counter.most_common(max_tokens)]
    token_to_idx = {tok: i for i, tok in enumerate(vocab)}

    X = np.zeros((len(selfies_list), len(vocab)), dtype=float)

    for i, s in enumerate(selfies_list):
        counts = Counter(sf.split_selfies(s))
        for tok, count in counts.items():
            if tok in token_to_idx:
                X[i, token_to_idx[tok]] = count

    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

    return X, vocab


X_latent, vocab = selfies_to_latent(
    df["selfies"].tolist(),
    max_tokens=MAX_SELFIES_TOKENS
)

print("Latent shape:", X_latent.shape)
print("Vocab size:", len(vocab))

Latent shape: (636, 27)
Vocab size: 27


In [11]:
# ============================================================
# CELL 8 — Base KMeans Molecular Bins
# ============================================================

N_QUBITS = BASE_N_QUBITS
N_BINS = BASE_N_BINS

kmeans = KMeans(
    n_clusters=N_BINS,
    random_state=SEED,
    n_init=20
)

df["bin"] = kmeans.fit_predict(X_latent)

bin_counts = df["bin"].value_counts().sort_index()
p_target = np.zeros(N_BINS, dtype=float)

for i in range(N_BINS):
    p_target[i] = bin_counts.get(i, 0)

p_target = p_target / p_target.sum()

print("Target mass:", p_target.sum())
print("Bins:", N_BINS)
print("Min bin count:", int(bin_counts.min()))
print("Max bin count:", int(bin_counts.max()))
print("Mean molecules per bin:", len(df) / N_BINS)

Target mass: 1.0
Bins: 32
Min bin count: 5
Max bin count: 48
Mean molecules per bin: 19.875


In [12]:
# ============================================================
# CELL 9 — Distribution Metrics
# ============================================================

def normalize(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-12, None)
    return p / p.sum()


def kl_divergence(p, q):
    p, q = normalize(p), normalize(q)
    return float(np.sum(p * np.log(p / q)))


def total_variation(p, q):
    p, q = normalize(p), normalize(q)
    return float(0.5 * np.sum(np.abs(p - q)))


def fidelity(p, q):
    p, q = normalize(p), normalize(q)
    return float(np.sum(np.sqrt(p * q)) ** 2)


def mmd_rbf(p, q, gammas=(0.1, 1.0, 10.0)):
    p, q = normalize(p), normalize(q)
    x = np.arange(len(p)).reshape(-1, 1)

    K = np.zeros((len(p), len(p)), dtype=float)

    for g in gammas:
        dist2 = (x - x.T) ** 2
        K += np.exp(-g * dist2)

    K /= len(gammas)

    diff = p - q
    return float(diff.T @ K @ diff)


def sinkhorn_1d(p, q):
    p, q = normalize(p), normalize(q)
    x = np.arange(len(p))
    return float(wasserstein_distance(x, x, p, q))


def distribution_report(name, p, target):
    return {
        "method": name,
        "KL(ref||test)": kl_divergence(target, p),
        "TV": total_variation(target, p),
        "Fidelity": fidelity(target, p),
        "MMD": mmd_rbf(target, p),
        "Sinkhorn/Wasserstein": sinkhorn_1d(target, p),
    }

In [13]:
# ============================================================
# CELL 10 — Latent Mapping for Generated Molecules
# ============================================================

def smiles_to_latent_using_vocab(smiles_list, vocab):
    token_to_idx = {tok: i for i, tok in enumerate(vocab)}
    X = np.zeros((len(smiles_list), len(vocab)), dtype=float)

    for i, smi in enumerate(smiles_list):
        try:
            selfies = sf.encoder(smi)
            counts = Counter(sf.split_selfies(selfies))
            for tok, count in counts.items():
                if tok in token_to_idx:
                    X[i, token_to_idx[tok]] = count
        except Exception:
            pass

    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return X


def distribution_from_generated_smiles(smiles_list, fitted_kmeans=kmeans, fitted_vocab=vocab, n_bins=N_BINS):
    valid = [s for s in smiles_list if valid_smiles(s)]

    if len(valid) == 0:
        return np.ones(n_bins) / n_bins

    X_gen = smiles_to_latent_using_vocab(valid, fitted_vocab)
    bins = fitted_kmeans.predict(X_gen)

    counts = np.zeros(n_bins, dtype=float)

    for b in bins:
        counts[int(b)] += 1

    return counts / counts.sum()


def distribution_from_bins(sampled_bins, n_bins):
    counts = np.bincount(np.asarray(sampled_bins, dtype=int), minlength=n_bins).astype(float)
    if counts.sum() == 0:
        return np.ones(n_bins) / n_bins
    return counts / counts.sum()

In [14]:
# ============================================================
# CELL 11 — Molecular Quality Metrics
# ============================================================

def mol_fingerprint(smiles, radius=2, n_bits=2048):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    except Exception:
        return None


def molecular_quality_report(name, gen_df, train_df):
    gen_smiles = gen_df["smiles"].dropna().astype(str).tolist()

    train_smiles = train_df["smiles"].dropna().astype(str).tolist()
    train_canon = []

    for s in train_smiles:
        _, canon, ok = sanitize_smiles(s)
        if ok:
            train_canon.append(canon)

    train_set = set(train_canon)

    valid = []
    for s in gen_smiles:
        _, canon, ok = sanitize_smiles(s)
        if ok:
            valid.append(canon)

    unique = sorted(set(valid))
    novel = [s for s in unique if s not in train_set]

    train_fps = [mol_fingerprint(s) for s in train_set]
    train_fps = [fp for fp in train_fps if fp is not None]

    max_sims = []
    for s in unique:
        fp = mol_fingerprint(s)
        if fp is None or len(train_fps) == 0:
            continue
        sims = [TanimotoSimilarity(fp, ref) for ref in train_fps]
        max_sims.append(max(sims))

    syba_scores = []
    if "syba_score" in gen_df.columns:
        syba_scores = gen_df["syba_score"].dropna().tolist()

    return {
        "method": name,
        "generated_total": len(gen_smiles),
        "valid_count": len(valid),
        "validity": len(valid) / max(len(gen_smiles), 1),
        "unique_count": len(unique),
        "uniqueness": len(unique) / max(len(valid), 1),
        "novel_count": len(novel),
        "novelty": len(novel) / max(len(unique), 1),
        "mean_SYBA": float(np.nanmean(syba_scores)) if len(syba_scores) else np.nan,
        "median_SYBA": float(np.nanmedian(syba_scores)) if len(syba_scores) else np.nan,
        "mean_max_train_Tanimoto": float(np.mean(max_sims)) if len(max_sims) else np.nan
    }


def full_generator_report(name, gen_df):
    p_gen = distribution_from_generated_smiles(gen_df["smiles"].tolist())
    dist = distribution_report(name, p_gen, p_target)
    qual = molecular_quality_report(name, gen_df, df)
    return {**dist, **qual}

In [15]:
# ============================================================
# CELL 12 — STONED SELFIES Baseline
# ============================================================

def get_selfie_chars(selfie):
    return list(sf.split_selfies(selfie))


def mutate_selfie(selfie, max_molecules_len):
    chars_selfie = get_selfie_chars(selfie)
    alphabet = list(sf.get_semantic_robust_alphabet())

    while True:
        random_choice = np.random.choice([1, 2, 3])

        if random_choice == 1:
            random_index = np.random.randint(len(chars_selfie) + 1)
            random_character = np.random.choice(alphabet)
            mutated_chars = chars_selfie[:random_index] + [random_character] + chars_selfie[random_index:]

        elif random_choice == 2:
            random_index = np.random.randint(len(chars_selfie))
            random_character = np.random.choice(alphabet)
            mutated_chars = chars_selfie[:random_index] + [random_character] + chars_selfie[random_index + 1:]

        else:
            if len(chars_selfie) <= 1:
                continue
            random_index = np.random.randint(len(chars_selfie))
            mutated_chars = chars_selfie[:random_index] + chars_selfie[random_index + 1:]

        mutated_selfie = "".join(mutated_chars)

        try:
            smiles = sf.decoder(mutated_selfie)
            mol, smiles_canon, done = sanitize_smiles(smiles)

            if len(mutated_chars) > max_molecules_len or smiles_canon == "":
                done = False

            if done:
                return mutated_selfie, smiles_canon
        except Exception:
            pass


def depth_n_selfies_mutation(selfies, depth):
    origin_selfies = selfies
    len_origin = len(get_selfie_chars(origin_selfies))
    current = origin_selfies

    for _ in range(depth):
        current, _ = mutate_selfie(current, len_origin + depth)

    return current


def classical_stoned_generate(df_in, mutations_per_molecule=1, max_depth=5):
    start = time.perf_counter()
    records = []

    for _, row in df_in.iterrows():
        base_selfies = row["selfies"]

        for _ in range(mutations_per_molecule):
            depth = random.randint(1, max_depth)

            try:
                mutated_selfies = depth_n_selfies_mutation(base_selfies, depth)
                mutated_smiles = safe_decode_selfies(mutated_selfies)

                if mutated_smiles is not None:
                    records.append({
                        "smiles": mutated_smiles,
                        "selfies": mutated_selfies,
                        "syba_score": syba_score_smiles(mutated_smiles),
                        "method": "Classical_STONED"
                    })
            except Exception:
                pass

    elapsed = time.perf_counter() - start
    return pd.DataFrame(records), elapsed


df_stoned, stoned_time = classical_stoned_generate(
    df,
    mutations_per_molecule=N_CLASSICAL_MUTATIONS_PER_MOLECULE,
    max_depth=STONED_MAX_DEPTH
)

print("STONED generated:", len(df_stoned))
print("STONED time:", stoned_time)
df_stoned.head()

STONED generated: 636
STONED time: 4.190495899994858


,smiles,selfies,syba_score,method
0,Cc1ccccccc(O)cc1C1N=Cc2c1nc(OCC1CCCN1C)nc2N1CC...,[C][N][C][C][C][C][Ring1][Branch1][C][O][C][=N...,-38.212254,Classical_STONED
1,CC(N=COCC1CCCN1C)N1CC2CCC(C1)N2C1C=NC(c2cc(O)c...,[C][N][C][C][C][C][Ring1][Branch1][C][O][C][=N...,-57.222085,Classical_STONED
2,Cn1ccnc1CCOC=NC(C(O)=Cc1c#cccc1)N1CCC2CC21,[C][N][C][=C][N][=C][Ring1][Branch1][C][C][O][...,-75.400122,Classical_STONED
3,C=C1N=C([O+]OCCc2ccccn2)N=C(C=CN=CC(F)C=CO)N2C...,[O][C][=C][C][Branch2][Ring2][#C][C][=N][C][=C...,-95.929423,Classical_STONED
4,CC(C=CC1(F)N=CC=C(N=COCc2ccnn2C)N2CC3CCC(N3)C2...,[C][N][N][=C][C][=C][Ring1][Branch1][C][O][C][...,-89.050636,Classical_STONED


In [16]:
# ============================================================
# CELL 13 — Random SELFIES Baseline
# ============================================================

def random_selfies_generate(n_samples, min_len=10, max_len=80):
    alphabet = list(sf.get_semantic_robust_alphabet())
    records = []
    start = time.perf_counter()

    for _ in range(n_samples):
        length = random.randint(min_len, max_len)
        selfie = "".join(np.random.choice(alphabet, size=length))
        smi = safe_decode_selfies(selfie)

        if smi is not None:
            records.append({
                "smiles": smi,
                "selfies": selfie,
                "syba_score": syba_score_smiles(smi),
                "method": "Random_SELFIES"
            })

    elapsed = time.perf_counter() - start
    return pd.DataFrame(records), elapsed


df_random_selfies, random_selfies_time = random_selfies_generate(N_GENERATED_MOLECULES)

print("Random SELFIES generated:", len(df_random_selfies))
print("Random SELFIES time:", random_selfies_time)
df_random_selfies.head()

Random SELFIES generated: 2000
Random SELFIES time: 0.818007599998964


,smiles,selfies,syba_score,method
0,F[B-]=[S+]=P[O+]=[S-][P+],[=P+1][S-1][=O+1][P][=S+1][=B-1][F][P-1][F][#P...,-2.379323,Random_SELFIES
1,B#[P-][B-]B=BC[C+][C-][B+]B[P+]BP,[=P][B][P+1][B][B+1][=C-1][C+1][C][B][#B][B-1]...,-4.901018,Random_SELFIES
2,[B+][O+]Br,[=B+1][O+1][Br][#Branch3][F][P+1][Ring1][#B-1]...,5.758512,Random_SELFIES
3,[S+]#[SH]1[N+][N+][P+][S+]=[S-]#S1,[N+1][N+1][P+1][S+1][=S-1][#S][S][Ring3][=Ring...,-4.348046,Random_SELFIES
4,N[N-][P-]F,[F][=P-1][N-1][N][Ring3][#B-1][=Ring3][=O][Rin...,1.547469,Random_SELFIES


In [17]:
# ============================================================
# CELL 14 — Empirical Cluster Retrieval Baseline
# ============================================================

def empirical_cluster_retrieval_generate(df_in, p_ref, n_samples):
    records = []
    start = time.perf_counter()

    sampled_bins = np.random.choice(
        np.arange(len(p_ref)),
        size=n_samples,
        p=normalize(p_ref)
    )

    for b in sampled_bins:
        pool = df_in[df_in["bin"] == int(b)]

        if len(pool) == 0:
            pool = df_in

        row = pool.sample(1, random_state=random.randint(0, 10**9)).iloc[0]

        records.append({
            "smiles": row["smiles"],
            "selfies": row["selfies"],
            "syba_score": syba_score_smiles(row["smiles"]),
            "method": "Empirical_Cluster_Retrieval"
        })

    elapsed = time.perf_counter() - start
    return pd.DataFrame(records), elapsed


df_retrieval, retrieval_time = empirical_cluster_retrieval_generate(
    df,
    p_target,
    N_GENERATED_MOLECULES
)

print("Retrieval generated:", len(df_retrieval))
print("Retrieval time:", retrieval_time)
df_retrieval.head()

Retrieval generated: 2000
Retrieval time: 4.545703100011451


,smiles,selfies,syba_score,method
0,Cc1ccc2[nH]ncc2c1-c1ncc2c(N3CC4CCC(C3)N4)nc(OC...,[C][C][=C][C][=C][NH1][N][=C][C][Ring1][Branch...,-41.012101,Empirical_Cluster_Retrieval
1,CC1CN2CCCC2(COc2nc(N3CC4CCC(C3)N4)c3cnc(-c4ccc...,[C][C][C][N][C][C][C][C][Ring1][Branch1][Branc...,-29.506641,Empirical_Cluster_Retrieval
2,CN1CCCC1COc1nc(N2CC3CCC(C2)N3)c2cnc(-c3cccc4cc...,[C][N][C][C][C][C][Ring1][Branch1][C][O][C][=N...,-10.352974,Empirical_Cluster_Retrieval
3,Oc1cc(-c2ncc3c(N4CC5CCC(C4)N5)nc(OCC4(C(F)(F)F...,[O][C][=C][C][Branch2][Branch1][=Branch1][C][=...,-22.594733,Empirical_Cluster_Retrieval
4,CN1CCCC1COc1nc(N2CC3CCC(C2)N3)c2cc(Oc3cccc(F)c...,[C][N][C][C][C][C][Ring1][Branch1][C][O][C][=N...,-1.880417,Empirical_Cluster_Retrieval


In [18]:
# ============================================================
# CELL 15 — Classical Born Machine Baselines
# ============================================================

def int_to_bits(x, n_qubits):
    return np.array(list(format(int(x), f"0{n_qubits}b")), dtype=int)


def bits_to_int(bits):
    return int("".join(str(int(b)) for b in bits), 2)


def train_independent_bernoulli_born(p_data, n_qubits):
    n_bins = 2 ** n_qubits
    bit_probs = np.zeros(n_qubits, dtype=float)

    for state in range(n_bins):
        bits = int_to_bits(state, n_qubits)
        bit_probs += p_data[state] * bits

    return np.clip(bit_probs, 1e-6, 1 - 1e-6)


def sample_independent_bernoulli(bit_probs, n_samples):
    n_qubits = len(bit_probs)
    sampled = []

    for _ in range(n_samples):
        bits = np.random.binomial(1, bit_probs, size=n_qubits)
        sampled.append(bits_to_int(bits))

    return np.array(sampled, dtype=int)


def train_markov_chain_born(p_data, n_qubits):
    n_bins = 2 ** n_qubits
    p0 = 0.0
    trans_counts = np.ones((n_qubits - 1, 2, 2), dtype=float) * 1e-6

    for state in range(n_bins):
        bits = int_to_bits(state, n_qubits)
        weight = p_data[state]
        p0 += weight * bits[0]

        for i in range(n_qubits - 1):
            trans_counts[i, bits[i], bits[i + 1]] += weight

    trans_probs = trans_counts / trans_counts.sum(axis=2, keepdims=True)
    return np.clip(p0, 1e-6, 1 - 1e-6), trans_probs


def sample_markov_chain_born(model, n_samples):
    p0, trans_probs = model
    n_qubits = trans_probs.shape[0] + 1
    sampled = []

    for _ in range(n_samples):
        bits = np.zeros(n_qubits, dtype=int)
        bits[0] = np.random.binomial(1, p0)

        for i in range(n_qubits - 1):
            bits[i + 1] = np.random.choice([0, 1], p=trans_probs[i, bits[i]])

        sampled.append(bits_to_int(bits))

    return np.array(sampled, dtype=int)


ind_model = train_independent_bernoulli_born(p_target, N_QUBITS)
ind_bins = sample_independent_bernoulli(ind_model, SHOTS)
p_independent_born = distribution_from_bins(ind_bins, N_BINS)

markov_model = train_markov_chain_born(p_target, N_QUBITS)
markov_bins = sample_markov_chain_born(markov_model, SHOTS)
p_markov_born = distribution_from_bins(markov_bins, N_BINS)

classical_born_reports = [
    distribution_report("Classical_Born_Independent_Bits", p_independent_born, p_target),
    distribution_report("Classical_Born_Markov_Chain", p_markov_born, p_target)
]

pd.DataFrame(classical_born_reports)

,method,KL(ref||test),TV,Fidelity,MMD,Sinkhorn/Wasserstein
0,Classical_Born_Independent_Bits,0.098435,0.177075,0.949363,0.004621,0.609355
1,Classical_Born_Markov_Chain,0.104514,0.178877,0.944645,0.003751,0.412189


In [19]:
# ============================================================
# CELL 16 — QCBM Circuit
# ============================================================

def build_logical_qcbm_circuit(theta, n_qubits, layers, measure=False):
    qc = QuantumCircuit(n_qubits)

    idx = 0

    for _ in range(layers):
        for q in range(n_qubits):
            qc.rz(theta[idx], q)
            idx += 1
            qc.sx(q)

            qc.rz(theta[idx], q)
            idx += 1
            qc.sx(q)

            qc.rz(theta[idx], q)
            idx += 1

        for q in range(n_qubits - 1):
            qc.cz(q, q + 1)

    if measure:
        qc.measure_all(add_bits=True)
        qc.data = [inst for inst in qc.data if inst.operation.name != "barrier"]

    return qc


def ideal_probs_from_theta(theta, n_qubits=N_QUBITS, layers=QCBM_LAYERS):
    qc = build_logical_qcbm_circuit(theta, n_qubits, layers, measure=False)
    sv = Statevector.from_instruction(qc)
    return normalize(np.real(sv.probabilities()))


def qcbm_loss(theta, p_ref=p_target, n_qubits=N_QUBITS, layers=QCBM_LAYERS):
    p_model = ideal_probs_from_theta(theta, n_qubits, layers)
    return mmd_rbf(p_ref, p_model) + 0.2 * total_variation(p_ref, p_model)


N_PARAMS = QCBM_LAYERS * N_QUBITS * 3
theta0 = np.random.uniform(-0.1, 0.1, size=N_PARAMS)

print("QCBM parameters:", N_PARAMS)
print("Initial loss:", qcbm_loss(theta0))

QCBM parameters: 45
Initial loss: 1.144103007390111


In [20]:
# ============================================================
# CELL 17 — Train Ideal QCBM
# ============================================================

start = time.perf_counter()

result = minimize(
    qcbm_loss,
    theta0,
    method="COBYLA",
    options={
        "maxiter": OPT_MAXITER,
        "rhobeg": 0.5,
        "tol": 1e-5,
        "disp": True
    }
)

qcbm_train_time = time.perf_counter() - start

theta_star = result.x
p_ideal_qcbm = ideal_probs_from_theta(theta_star)

ideal_qcbm_metrics = distribution_report(
    "Ideal_QCBM_Statevector",
    p_ideal_qcbm,
    p_target
)

print("Training time:", qcbm_train_time)
print("Final loss:", qcbm_loss(theta_star))
ideal_qcbm_metrics

Return from COBYLA because the objective function has been evaluated MAXFUN times.
Number of function values = 250   Least value of F = 0.0332481906936999
The corresponding X is:
[ 0.50207019  0.59245697  0.02331819  1.06888175  0.86130954 -0.2504025
  0.14593839  0.42017547  0.31396138  0.09252229  0.99089028 -0.06131427
  0.70249766  0.24585045  0.28244892  0.59628643  1.37422305  0.03265794
  0.31036966 -0.18855057  0.88310618 -0.05087064  0.15676476  0.1526596
  0.36184642 -0.32055053  0.83423568  0.15268727  1.22254395  0.4046191
  0.14202055 -0.34144411  0.72079818  1.25664624  1.16333585 -0.05486903
 -0.02303566  1.35950866 -0.15600793  0.51096947  1.02438758  0.84673232
  0.62080542 -1.04365952 -0.55616209]

Training time: 2.7988621999975294
Final loss: 0.0332481906936999


{'method': 'Ideal_QCBM_Statevector',
 'KL(ref||test)': 0.09404340288707187,
 'TV': 0.1467821252248913,
 'Fidelity': 0.9501985720853626,
 'MMD': 0.003891765648721638,
 'Sinkhorn/Wasserstein': 0.3873608706113019}

In [21]:
# ============================================================
# CELL 18 — Ideal QCBM Sampling
# ============================================================

start = time.perf_counter()

ideal_sampled_bins = np.random.choice(
    np.arange(N_BINS),
    size=SHOTS,
    p=normalize(p_ideal_qcbm)
)

ideal_counts = Counter(
    format(int(b), f"0{N_QUBITS}b")
    for b in ideal_sampled_bins
)

ideal_sampling_time = time.perf_counter() - start
p_ideal_sampled = distribution_from_bins(ideal_sampled_bins, N_BINS)

ideal_sampled_metrics = distribution_report(
    "Ideal_QCBM_Sampled",
    p_ideal_sampled,
    p_target
)

print("Ideal sampling time:", ideal_sampling_time)
print("Sample counts:", list(ideal_counts.items())[:10])
ideal_sampled_metrics

Ideal sampling time: 0.0018678999913390726
Sample counts: [('00110', 87), ('11010', 101), ('00011', 79), ('10010', 70), ('00010', 117), ('11000', 88), ('11100', 71), ('10101', 27), ('01110', 62), ('10000', 74)]


{'method': 'Ideal_QCBM_Sampled',
 'KL(ref||test)': 0.1047000329493318,
 'TV': 0.15551572327044025,
 'Fidelity': 0.9466447946049334,
 'MMD': 0.004347920188113436,
 'Sinkhorn/Wasserstein': 0.39173270440251634}

In [22]:
# ============================================================
# CELL 19 — QCBM Molecule Retrieval
# ============================================================

def generate_molecules_from_bin_distribution(df_in, p_bins, n_samples, method_name):
    sampled_bins = np.random.choice(
        np.arange(len(p_bins)),
        size=n_samples,
        p=normalize(p_bins)
    )

    records = []

    for b in sampled_bins:
        pool = df_in[df_in["bin"] == int(b)]

        if len(pool) == 0:
            pool = df_in

        row = pool.sample(1, random_state=random.randint(0, 10**9)).iloc[0]

        records.append({
            "smiles": row["smiles"],
            "selfies": row["selfies"],
            "syba_score": syba_score_smiles(row["smiles"]),
            "method": method_name,
            "bin": int(b)
        })

    return pd.DataFrame(records)


df_qcbm_ideal_molecules = generate_molecules_from_bin_distribution(
    df,
    p_ideal_sampled,
    N_GENERATED_MOLECULES,
    "Ideal_QCBM_Molecule_Retrieval"
)

df_qcbm_ideal_molecules.head()

,smiles,selfies,syba_score,method,bin
0,CN1CCCC1COc1nc(N2CC3CCC(C2)N3)c2cc(Oc3ccccc3C#...,[C][N][C][C][C][C][Ring1][Branch1][C][O][C][=N...,-7.201249,Ideal_QCBM_Molecule_Retrieval,20
1,CN(C)C(=O)OCC1CCC2(COc3nc4c(c(N5CC6CCC(C5)N6)n...,[C][N][Branch1][C][C][C][=Branch1][C][=O][O][C...,-24.328086,Ideal_QCBM_Molecule_Retrieval,4
2,Fc1c(-c2cccc3cccc(OC(F)(F)F)c23)ncc2c(N3CC4CCC...,[F][C][=C][Branch2][Ring1][#Branch2][C][=C][C]...,-12.113147,Ideal_QCBM_Molecule_Retrieval,14
3,Cn1ccnc1COc1nc(N2CC3CCC(C2)N3)c2cnc(-c3cc(O)cc...,[C][N][C][=C][N][=C][Ring1][Branch1][C][O][C][...,1.457580,Ideal_QCBM_Molecule_Retrieval,20
4,OC1CC2CN(c3nc(OCCC(F)(F)F)nc4c(F)c(-c5cccc6ccc...,[O][C][C][C][C][N][Branch2][Branch1][Branch1][...,-13.212445,Ideal_QCBM_Molecule_Retrieval,22


In [23]:
# ============================================================
# CELL 20 — Qubit Scaling Study
# ============================================================

def build_binned_target(df_in, n_qubits, max_selfies_tokens=32, seed=SEED):
    n_bins = 2 ** n_qubits

    if n_bins > len(df_in):
        raise ValueError(
            f"{n_qubits} qubits require {n_bins} bins but only {len(df_in)} molecules are available."
        )

    X_local, vocab_local = selfies_to_latent(
        df_in["selfies"].tolist(),
        max_tokens=max_selfies_tokens
    )

    km = KMeans(n_clusters=n_bins, random_state=seed, n_init=20)
    labels = km.fit_predict(X_local)

    counts = np.bincount(labels, minlength=n_bins).astype(float)
    p_local = counts / counts.sum()

    report = {
        "n_qubits": n_qubits,
        "n_bins": n_bins,
        "molecules": len(df_in),
        "min_bin_count": int(counts.min()),
        "max_bin_count": int(counts.max()),
        "empty_bins": int(np.sum(counts == 0)),
        "mean_molecules_per_bin": float(len(df_in) / n_bins)
    }

    return X_local, vocab_local, km, labels, p_local, report


def train_ideal_qcbm_for_distribution(p_ref, n_qubits, layers=QCBM_LAYERS, maxiter=OPT_MAXITER):
    n_params = layers * n_qubits * 3
    theta_init = np.random.uniform(-0.1, 0.1, size=n_params)

    start = time.perf_counter()

    res = minimize(
        lambda th: qcbm_loss(th, p_ref=p_ref, n_qubits=n_qubits, layers=layers),
        theta_init,
        method="COBYLA",
        options={
            "maxiter": maxiter,
            "rhobeg": 0.5,
            "tol": 1e-5,
            "disp": False
        }
    )

    elapsed = time.perf_counter() - start
    p_model = ideal_probs_from_theta(res.x, n_qubits=n_qubits, layers=layers)

    return res.x, p_model, elapsed, res.fun


qcbm_sweep_reports = []

for nq in QUBIT_SWEEP:
    try:
        _, _, _, _, p_ref_nq, sparse_report = build_binned_target(
            df,
            nq,
            MAX_SELFIES_TOKENS
        )

        theta_nq, p_qcbm_nq, train_time_nq, loss_nq = train_ideal_qcbm_for_distribution(
            p_ref_nq,
            n_qubits=nq,
            layers=QCBM_LAYERS,
            maxiter=OPT_MAXITER
        )

        report = distribution_report(
            f"Ideal_QCBM_{nq}_Qubits",
            p_qcbm_nq,
            p_ref_nq
        )

        report.update(sparse_report)
        report["training_time_sec"] = train_time_nq
        report["final_loss"] = loss_nq
        report["params"] = QCBM_LAYERS * nq * 3

        qcbm_sweep_reports.append(report)

    except Exception as e:
        qcbm_sweep_reports.append({
            "method": f"Ideal_QCBM_{nq}_Qubits",
            "n_qubits": nq,
            "error": str(e)
        })

qcbm_sweep_df = pd.DataFrame(qcbm_sweep_reports)
qcbm_sweep_df

,method,KL(ref||test),TV,Fidelity,MMD,Sinkhorn/Wasserstein,n_qubits,n_bins,molecules,min_bin_count,max_bin_count,empty_bins,mean_molecules_per_bin,training_time_sec,final_loss,params
0,Ideal_QCBM_5_Qubits,0.094363,0.143030,0.950519,0.003914,0.385610,5,32,636,5,48,0,19.875000,2.988636,0.032520,45
1,Ideal_QCBM_6_Qubits,0.184964,0.231210,0.907606,0.005973,1.012665,6,64,636,2,35,0,9.937500,3.181512,0.052215,54
2,Ideal_QCBM_7_Qubits,0.186316,0.235985,0.907398,0.002878,2.016185,7,128,636,1,19,0,4.968750,3.480889,0.050075,63
3,Ideal_QCBM_8_Qubits,0.246163,0.276465,0.885802,0.002871,1.526088,8,256,636,1,15,0,2.484375,5.004676,0.058164,72


In [26]:
# ============================================================
# CELL 21 — Hardware Backend Selection
# ============================================================

if RUN_IBM_HARDWARE:
    service = QiskitRuntimeService(name=IBM_ACCOUNT_NAME)

    available_backends = service.backends()
    backend_names = [b.name for b in available_backends]

    print("Available backends:", backend_names)

    if BACKEND_NAME in backend_names:
        backend = service.backend(BACKEND_NAME)
    else:
        suitable = [b for b in available_backends if b.num_qubits >= N_QUBITS]
        suitable.sort(key=lambda b: b.status().pending_jobs)
        backend = suitable[0]

    print("Chosen backend:", backend.name)
    print("Qubits:", backend.num_qubits)
    print("Pending jobs:", backend.status().pending_jobs)
    print("Basis gates:", backend.operation_names)
else:
    backend = None
    print("Hardware execution disabled. Set RUN_IBM_HARDWARE = True to run this cell.")

Available backends: ['ibm_fez', 'ibm_marrakesh', 'ibm_kingston']
Chosen backend: ibm_fez
Qubits: 156
Pending jobs: 4
Basis gates: ['delay', 'if_else', 'reset', 'x', 'sx', 'id', 'measure', 'cz', 'rz']


In [27]:
# ============================================================
# CELL 22 — Hardware CZ Connectivity and Physical Qubits
# ============================================================

def extract_cz_edges(backend):
    backend_target = backend.target
    edges = []

    for inst_key in backend_target:
        if inst_key == "cz":
            for qargs in backend_target[inst_key]:
                if qargs is not None and len(qargs) == 2:
                    edges.append(tuple(qargs))

    return sorted(set(edges))


def find_connected_path(edges, length):
    graph = defaultdict(list)

    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)

    for start in graph:
        stack = [(start, [start])]

        while stack:
            node, path = stack.pop()

            if len(path) == length:
                return path

            for nxt in graph[node]:
                if nxt not in path:
                    stack.append((nxt, path + [nxt]))

    return None


if RUN_IBM_HARDWARE:
    cz_edges = extract_cz_edges(backend)
    physical_qubits = find_connected_path(cz_edges, N_QUBITS)

    if physical_qubits is None:
        raise RuntimeError("Could not find hardware-compatible connected qubit path.")

    hardware_edges = [
        (physical_qubits[i], physical_qubits[i + 1])
        for i in range(len(physical_qubits) - 1)
    ]

    print("Selected physical qubits:", physical_qubits)
    print("Hardware CZ edges:", hardware_edges)
else:
    physical_qubits = None

Selected physical qubits: [0, 1, 2, 3, 16]
Hardware CZ edges: [(0, 1), (1, 2), (2, 3), (3, 16)]


In [28]:
# ============================================================
# CELL 23 — Build Real Hardware Circuit
# ============================================================

def build_real_qcbm_circuit(theta, backend, physical_qubits, layers=QCBM_LAYERS):
    qc = QuantumCircuit(backend.num_qubits, len(physical_qubits))

    idx = 0

    for _ in range(layers):
        for q in physical_qubits:
            qc.rz(theta[idx], q)
            idx += 1
            qc.sx(q)

            qc.rz(theta[idx], q)
            idx += 1
            qc.sx(q)

            qc.rz(theta[idx], q)
            idx += 1

        for i in range(len(physical_qubits) - 1):
            qc.cz(physical_qubits[i], physical_qubits[i + 1])

    for c, q in enumerate(physical_qubits):
        qc.measure(q, c)

    return qc


if RUN_IBM_HARDWARE:
    qc_real = build_real_qcbm_circuit(
        theta_star,
        backend,
        physical_qubits,
        layers=QCBM_LAYERS
    )

    print("Hardware circuit depth:", qc_real.depth())
    print("Hardware circuit ops:", qc_real.count_ops())
else:
    qc_real = None

Hardware circuit depth: 24
Hardware circuit ops: OrderedDict([('rz', 45), ('sx', 30), ('cz', 12), ('measure', 5)])


In [29]:
# ============================================================
# CELL 24 — Submit IBM Hardware Job
# ============================================================

if RUN_IBM_HARDWARE:
    sampler_real = Sampler(mode=backend)

    start_real = time.perf_counter()

    job_real = sampler_real.run(
        [qc_real],
        shots=SHOTS
    )

    print("Submitted job ID:", job_real.job_id())
    print("Backend:", backend.name)
    print("Shots:", SHOTS)
    print("Physical qubits:", physical_qubits)
else:
    job_real = None
    start_real = None
    print("Hardware job not submitted.")

Submitted job ID: d8l92bbqv2lc73866i1g
Backend: ibm_fez
Shots: 2000
Physical qubits: [0, 1, 2, 3, 16]


In [30]:
# ============================================================
# CELL 25 — Fetch IBM Hardware Result
# ============================================================

def counts_to_probability(counts, n_qubits):
    p = np.zeros(2 ** n_qubits, dtype=float)

    for bitstr, count in counts.items():
        clean = bitstr.replace(" ", "")
        idx = int(clean, 2)

        if idx < len(p):
            p[idx] += count

    if p.sum() == 0:
        return np.ones(len(p)) / len(p)

    return p / p.sum()


def extract_sampler_bitstrings(result):
    pub_result = result[0]
    data = pub_result.data

    if hasattr(data, "c"):
        return data.c.get_bitstrings()

    if hasattr(data, "meas"):
        return data.meas.get_bitstrings()

    for name in dir(data):
        obj = getattr(data, name)
        if hasattr(obj, "get_bitstrings"):
            return obj.get_bitstrings()

    raise RuntimeError("Could not extract bitstrings from SamplerV2 result.")


if RUN_IBM_HARDWARE:
    result_real = job_real.result()
    bitstrings_real = extract_sampler_bitstrings(result_real)
    counts_real = Counter(bitstrings_real)

    real_time = time.perf_counter() - start_real
    p_real = counts_to_probability(counts_real, N_QUBITS)

    real_metrics = distribution_report(
        f"Real_Hardware_{backend.name}",
        p_real,
        p_target
    )

    print("Real hardware time:", real_time)
    print("Real count sample:", list(counts_real.items())[:10])
    real_metrics
else:
    p_real = None
    real_metrics = None
    real_time = None
    print("No hardware result.")

Real hardware time: 398.91557640000246
Real count sample: [('10000', 79), ('10011', 48), ('11000', 91), ('01010', 70), ('10111', 46), ('00011', 90), ('11001', 21), ('00000', 115), ('00100', 87), ('10010', 79)]


In [31]:
# ============================================================
# CELL 26 — Hardware Molecule Retrieval
# ============================================================

if RUN_IBM_HARDWARE:
    df_qcbm_hardware_molecules = generate_molecules_from_bin_distribution(
        df,
        p_real,
        N_GENERATED_MOLECULES,
        f"Hardware_QCBM_{backend.name}_Molecule_Retrieval"
    )

    df_qcbm_hardware_molecules.head()
else:
    df_qcbm_hardware_molecules = pd.DataFrame()
    print("Hardware molecule retrieval skipped.")

In [32]:
# ============================================================
# CELL 27 — Unified Benchmark Table
# ============================================================

benchmark_rows = []

benchmark_rows.append({
    **distribution_report("Classical_STONED", distribution_from_generated_smiles(df_stoned["smiles"].tolist()), p_target),
    **molecular_quality_report("Classical_STONED", df_stoned, df),
    "runtime_sec": stoned_time
})

benchmark_rows.append({
    **distribution_report("Random_SELFIES", distribution_from_generated_smiles(df_random_selfies["smiles"].tolist()), p_target),
    **molecular_quality_report("Random_SELFIES", df_random_selfies, df),
    "runtime_sec": random_selfies_time
})

benchmark_rows.append({
    **distribution_report("Empirical_Cluster_Retrieval", distribution_from_generated_smiles(df_retrieval["smiles"].tolist()), p_target),
    **molecular_quality_report("Empirical_Cluster_Retrieval", df_retrieval, df),
    "runtime_sec": retrieval_time
})

benchmark_rows.append({
    **distribution_report("Classical_Born_Independent_Bits", p_independent_born, p_target),
    "runtime_sec": np.nan
})

benchmark_rows.append({
    **distribution_report("Classical_Born_Markov_Chain", p_markov_born, p_target),
    "runtime_sec": np.nan
})

benchmark_rows.append({
    **distribution_report("Ideal_QCBM_Statevector", p_ideal_qcbm, p_target),
    "runtime_sec": qcbm_train_time
})

benchmark_rows.append({
    **distribution_report("Ideal_QCBM_Sampled", p_ideal_sampled, p_target),
    **molecular_quality_report("Ideal_QCBM_Molecule_Retrieval", df_qcbm_ideal_molecules, df),
    "runtime_sec": ideal_sampling_time
})

if RUN_IBM_HARDWARE and real_metrics is not None:
    benchmark_rows.append({
        **real_metrics,
        **molecular_quality_report(
            f"Hardware_QCBM_{backend.name}_Molecule_Retrieval",
            df_qcbm_hardware_molecules,
            df
        ),
        "runtime_sec": real_time
    })

benchmark_df = pd.DataFrame(benchmark_rows)

preferred_cols = [
    "method",
    "KL(ref||test)",
    "TV",
    "Fidelity",
    "MMD",
    "Sinkhorn/Wasserstein",
    "validity",
    "uniqueness",
    "novelty",
    "mean_SYBA",
    "median_SYBA",
    "mean_max_train_Tanimoto",
    "generated_total",
    "valid_count",
    "unique_count",
    "novel_count",
    "runtime_sec"
]

benchmark_df = benchmark_df[[c for c in preferred_cols if c in benchmark_df.columns]]
benchmark_df.sort_values(["KL(ref||test)", "TV"], ascending=True)

,method,KL(ref||test),TV,Fidelity,MMD,Sinkhorn/Wasserstein,validity,uniqueness,novelty,mean_SYBA,median_SYBA,mean_max_train_Tanimoto,generated_total,valid_count,unique_count,novel_count,runtime_sec
2,Empirical_Cluster_Retrieval,0.005732,0.042972,0.997149,0.000274,0.231393,1.0,0.2910,0.000000,-22.513840,-20.083113,1.000000,2000.0,2000.0,582.0,0.0,4.545703
5,Ideal_QCBM_Statevector,0.094043,0.146782,0.950199,0.003892,0.387361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.798862
3,Classical_Born_Independent_Bits,0.098435,0.177075,0.949363,0.004621,0.609355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Classical_Born_Markov_Chain,0.104514,0.178877,0.944645,0.003751,0.412189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Ideal_QCBM_Molecule_Retrieval,0.104700,0.155516,0.946645,0.004348,0.391733,1.0,0.2835,0.000000,-23.324070,-20.291090,1.000000,2000.0,2000.0,567.0,0.0,0.001868
7,Hardware_QCBM_ibm_fez_Molecule_Retrieval,0.130839,0.171689,0.936290,0.004103,0.334692,1.0,0.2790,0.000000,-23.680763,-20.667008,1.000000,2000.0,2000.0,558.0,0.0,398.915576
0,Classical_STONED,0.480432,0.366352,0.809089,0.018971,0.871069,1.0,1.0000,0.992138,-92.246130,-91.293787,0.282961,636.0,636.0,636.0,631.0,4.190496
1,Random_SELFIES,4.037279,0.679302,0.390363,0.184226,3.895629,1.0,0.8385,1.000000,0.851170,4.902203,0.031703,2000.0,2000.0,1677.0,1677.0,0.818008


In [33]:
# ============================================================
# CELL 28 — Save Results
# ============================================================

benchmark_path = RESULTS_DIR / "qcbm_vs_classical_benchmark.csv"
sweep_path = RESULTS_DIR / "qcbm_qubit_scaling_study.csv"

benchmark_df.to_csv(benchmark_path, index=False)
qcbm_sweep_df.to_csv(sweep_path, index=False)

df_stoned.to_csv(RESULTS_DIR / "generated_stoned.csv", index=False)
df_random_selfies.to_csv(RESULTS_DIR / "generated_random_selfies.csv", index=False)
df_retrieval.to_csv(RESULTS_DIR / "generated_empirical_retrieval.csv", index=False)
df_qcbm_ideal_molecules.to_csv(RESULTS_DIR / "generated_ideal_qcbm.csv", index=False)

if RUN_IBM_HARDWARE and len(df_qcbm_hardware_molecules):
    df_qcbm_hardware_molecules.to_csv(
        RESULTS_DIR / "generated_hardware_qcbm.csv",
        index=False
    )

print("Saved:")
print(benchmark_path)
print(sweep_path)

Saved:
RESULTS\qcbm_vs_classical_benchmark.csv
RESULTS\qcbm_qubit_scaling_study.csv


In [34]:
# ============================================================
# CELL 28 — Save Results
# ============================================================

benchmark_path = RESULTS_DIR / "qcbm_vs_classical_benchmark.csv"
sweep_path = RESULTS_DIR / "qcbm_qubit_scaling_study.csv"

benchmark_df.to_csv(benchmark_path, index=False)
qcbm_sweep_df.to_csv(sweep_path, index=False)

df_stoned.to_csv(RESULTS_DIR / "generated_stoned.csv", index=False)
df_random_selfies.to_csv(RESULTS_DIR / "generated_random_selfies.csv", index=False)
df_retrieval.to_csv(RESULTS_DIR / "generated_empirical_retrieval.csv", index=False)
df_qcbm_ideal_molecules.to_csv(RESULTS_DIR / "generated_ideal_qcbm.csv", index=False)

if RUN_IBM_HARDWARE and len(df_qcbm_hardware_molecules):
    df_qcbm_hardware_molecules.to_csv(
        RESULTS_DIR / "generated_hardware_qcbm.csv",
        index=False
    )

print("Saved:")
print(benchmark_path)
print(sweep_path)

Saved:
RESULTS\qcbm_vs_classical_benchmark.csv
RESULTS\qcbm_qubit_scaling_study.csv


In [35]:
# ============================================================
# CELL 29 — Final Summary Tables
# ============================================================

print("Main Benchmark")
display(benchmark_df.sort_values(["KL(ref||test)", "TV"], ascending=True))

print("Qubit Scaling")
display(qcbm_sweep_df)

Main Benchmark


,method,KL(ref||test),TV,Fidelity,MMD,Sinkhorn/Wasserstein,validity,uniqueness,novelty,mean_SYBA,median_SYBA,mean_max_train_Tanimoto,generated_total,valid_count,unique_count,novel_count,runtime_sec
2,Empirical_Cluster_Retrieval,0.005732,0.042972,0.997149,0.000274,0.231393,1.0,0.2910,0.000000,-22.513840,-20.083113,1.000000,2000.0,2000.0,582.0,0.0,4.545703
5,Ideal_QCBM_Statevector,0.094043,0.146782,0.950199,0.003892,0.387361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.798862
3,Classical_Born_Independent_Bits,0.098435,0.177075,0.949363,0.004621,0.609355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Classical_Born_Markov_Chain,0.104514,0.178877,0.944645,0.003751,0.412189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Ideal_QCBM_Molecule_Retrieval,0.104700,0.155516,0.946645,0.004348,0.391733,1.0,0.2835,0.000000,-23.324070,-20.291090,1.000000,2000.0,2000.0,567.0,0.0,0.001868
7,Hardware_QCBM_ibm_fez_Molecule_Retrieval,0.130839,0.171689,0.936290,0.004103,0.334692,1.0,0.2790,0.000000,-23.680763,-20.667008,1.000000,2000.0,2000.0,558.0,0.0,398.915576
0,Classical_STONED,0.480432,0.366352,0.809089,0.018971,0.871069,1.0,1.0000,0.992138,-92.246130,-91.293787,0.282961,636.0,636.0,636.0,631.0,4.190496
1,Random_SELFIES,4.037279,0.679302,0.390363,0.184226,3.895629,1.0,0.8385,1.000000,0.851170,4.902203,0.031703,2000.0,2000.0,1677.0,1677.0,0.818008


Qubit Scaling


,method,KL(ref||test),TV,Fidelity,MMD,Sinkhorn/Wasserstein,n_qubits,n_bins,molecules,min_bin_count,max_bin_count,empty_bins,mean_molecules_per_bin,training_time_sec,final_loss,params
0,Ideal_QCBM_5_Qubits,0.094363,0.143030,0.950519,0.003914,0.385610,5,32,636,5,48,0,19.875000,2.988636,0.032520,45
1,Ideal_QCBM_6_Qubits,0.184964,0.231210,0.907606,0.005973,1.012665,6,64,636,2,35,0,9.937500,3.181512,0.052215,54
2,Ideal_QCBM_7_Qubits,0.186316,0.235985,0.907398,0.002878,2.016185,7,128,636,1,19,0,4.968750,3.480889,0.050075,63
3,Ideal_QCBM_8_Qubits,0.246163,0.276465,0.885802,0.002871,1.526088,8,256,636,1,15,0,2.484375,5.004676,0.058164,72


In [36]:
# ============================================================
# CELL 30 — Interpretation Helper
# ============================================================

def interpret_qubit_scaling(sweep_df):
    cols = [
        "n_qubits",
        "n_bins",
        "mean_molecules_per_bin",
        "min_bin_count",
        "empty_bins",
        "KL(ref||test)",
        "TV",
        "Fidelity",
        "training_time_sec"
    ]

    available = [c for c in cols if c in sweep_df.columns]
    return sweep_df[available].sort_values("n_qubits")


interpret_qubit_scaling(qcbm_sweep_df)

,n_qubits,n_bins,mean_molecules_per_bin,min_bin_count,empty_bins,KL(ref||test),TV,Fidelity,training_time_sec
0,5,32,19.875000,5,0,0.094363,0.143030,0.950519,2.988636
1,6,64,9.937500,2,0,0.184964,0.231210,0.907606,3.181512
2,7,128,4.968750,1,0,0.186316,0.235985,0.907398,3.480889
3,8,256,2.484375,1,0,0.246163,0.276465,0.885802,5.004676
